In [ ]:
!pip install mne -q
import os, re, glob, random, subprocess, shutil
import numpy as np
import mne
from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/diss-datasets"
PARTS_DIR = os.path.join(SAVE_DIR, "chbmit_parts_v2")
SAVE_NAME_V2 = "CHBMIT_crosspatient_v2.npy"
DATA_ROOT = "physionet.org/files/chbmit/1.0.0"

FS = 256
WINDOW_SEC = 10
WINDOW = FS * WINDOW_SEC
GUARD_SEC = 120
NONSEIZURE_RATIO = 2.0
SEED = 0

TARGET = ["FP1-F7","F7-T7","T7-P7","P7-O1","FP1-F3","F3-C3","C3-P3","P3-O1",
          "FP2-F4","F4-C4","C4-P4","P4-O2","FP2-F8","F8-T8","T8-P8","P8-O2",
          "FZ-CZ","CZ-PZ"]

PATIENTS = [f"chb{i:02d}" for i in range(1, 25)]
TEST_PATIENTS = ["chb20","chb22","chb23","chb24"]

random.seed(SEED); np.random.seed(SEED)
os.makedirs(PARTS_DIR, exist_ok=True)
print("PARTS_DIR ready:", os.path.exists(PARTS_DIR))

In [ ]:
!pip install awscli -q

NEG_CAP = 3000

def parse_summary(path):
    seiz, cur, pending = {}, None, None
    for line in open(path):
        line = line.strip()
        if line.startswith("File Name"):
            cur = line.split(":", 1)[1].strip(); seiz.setdefault(cur, [])
        elif "Seizure" in line and "Start Time" in line:
            pending = int(re.findall(r"\d+", line.split(":")[-1])[0])
        elif "Seizure" in line and "End Time" in line and pending is not None:
            e = int(re.findall(r"\d+", line.split(":")[-1])[0])
            seiz[cur].append((pending, e)); pending = None
    return seiz

def pick_channels(raw):
    clean = {}
    for i, n in enumerate(raw.ch_names):
        c = re.sub(r"-\d+$", "", n.upper().replace(" ", ""))
        if c not in clean:
            clean[c] = i
    if all(t in clean for t in TARGET):
        return [clean[t] for t in TARGET]
    return None

def windows_from_file(path, seizures, patient, fname):
    raw = mne.io.read_raw_edf(path, preload=True, verbose="ERROR")
    idx = pick_channels(raw)
    if idx is None:
        return [], [], [], []
    if int(raw.info["sfreq"]) != FS:
        raw.resample(FS, verbose="ERROR")
    data = raw.get_data()[idx].astype(np.float32)
    del raw
    n = data.shape[1] // WINDOW
    pos, neg, pos_meta, neg_meta = [], [], [], []
    for i in range(n):
        t0, t1 = i * WINDOW_SEC, (i + 1) * WINDOW_SEC
        seg = data[:, i * WINDOW:(i + 1) * WINDOW].copy()
        is_seiz = any(s < t1 and t0 < e for s, e in seizures)
        near = any(t0 < e + GUARD_SEC and t1 > s - GUARD_SEC for s, e in seizures)
        meta = (patient, fname, t0)
        if is_seiz:
            pos.append(seg); pos_meta.append(meta)
        elif not near:
            neg.append(seg); neg_meta.append(meta)
    return pos, neg, pos_meta, neg_meta

def process_patient(p):
    pdir = os.path.join(DATA_ROOT, p)
    subprocess.run(["aws","s3","sync",
                    f"s3://physionet-open/chbmit/1.0.0/{p}/",
                    f"{pdir}/","--no-sign-request","--only-show-errors"],
                   timeout=1800)
    summ = os.path.join(pdir, f"{p}-summary.txt")
    if not os.path.exists(summ):
        return None
    seiz_map = parse_summary(summ)
    pos_all, neg_all, pos_meta_all, neg_meta_all, seen = [], [], [], [], 0
    for edf in sorted(glob.glob(os.path.join(pdir, "*.edf"))):
        fname = os.path.basename(edf)
        pos, neg, pos_meta, neg_meta = windows_from_file(edf, seiz_map.get(fname, []), p, fname)
        pos_all += pos
        pos_meta_all += pos_meta
        for w, m in zip(neg, neg_meta):
            if len(neg_all) < NEG_CAP:
                neg_all.append(w); neg_meta_all.append(m)
            else:
                j = random.randint(0, seen)
                if j < NEG_CAP:
                    neg_all[j] = w; neg_meta_all[j] = m
            seen += 1
    shutil.rmtree(pdir, ignore_errors=True)
    if len(pos_all) == 0:
        return None
    combined = list(zip(neg_all, neg_meta_all))
    random.shuffle(combined)
    neg_all, neg_meta_all = zip(*combined) if combined else ([], [])
    keep = int(NONSEIZURE_RATIO * len(pos_all))
    neg_all, neg_meta_all = list(neg_all[:keep]), list(neg_meta_all[:keep])

    X = np.asarray(pos_all + neg_all, np.float32)
    y = np.asarray([1]*len(pos_all) + [0]*len(neg_all), np.int64)
    meta_all = pos_meta_all + neg_meta_all
    patients = np.asarray([m[0] for m in meta_all])
    files = np.asarray([m[1] for m in meta_all])
    starts = np.asarray([m[2] for m in meta_all], np.int64)
    np.save(os.path.join(PARTS_DIR, f"{p}.npy"),
            {"X": X, "y": y, "patient": patients, "file": files, "t0": starts},
            allow_pickle=True)
    return len(pos_all), len(neg_all)

for p in PATIENTS:
    if os.path.exists(os.path.join(PARTS_DIR, f"{p}.npy")):
        print(p, "already done, skipping", flush=True); continue
    try:
        r = process_patient(p)
        print(p, "seizure/nonseizure", r, flush=True)
    except subprocess.TimeoutExpired:
        print(p, "download timed out, skipping", flush=True)

In [ ]:
Xtr, ytr, Xte, yte = [], [], [], []
ptr, ftr, ttr = [], [], []
pte, fte, tte = [], [], []
for p in PATIENTS:
    fp = os.path.join(PARTS_DIR, f"{p}.npy")
    if not os.path.exists(fp):
        continue
    d = np.load(fp, allow_pickle=True).item()
    if p in TEST_PATIENTS:
        Xte.append(d["X"]); yte.append(d["y"])
        pte.append(d["patient"]); fte.append(d["file"]); tte.append(d["t0"])
    else:
        Xtr.append(d["X"]); ytr.append(d["y"])
        ptr.append(d["patient"]); ftr.append(d["file"]); ttr.append(d["t0"])

def stack(Xs, ys, ps, fs, ts):
    if not Xs:
        return (np.zeros((0, len(TARGET), WINDOW), np.float32), np.zeros((0,), np.int64),
                np.zeros((0,), object), np.zeros((0,), object), np.zeros((0,), np.int64))
    X = np.concatenate(Xs); y = np.concatenate(ys)
    p = np.concatenate(ps); f = np.concatenate(fs); t = np.concatenate(ts)
    order = np.random.permutation(len(X))
    return X[order], y[order], p[order], f[order], t[order]

Xtr_s, ytr_s, ptr_s, ftr_s, ttr_s = stack(Xtr, ytr, ptr, ftr, ttr)
Xte_s, yte_s, pte_s, fte_s, tte_s = stack(Xte, yte, pte, fte, tte)

data = {
    "train": {"X": Xtr_s, "y": ytr_s, "patient": ptr_s, "file": ftr_s, "t0": ttr_s},
    "test":  {"X": Xte_s, "y": yte_s, "patient": pte_s, "file": fte_s, "t0": tte_s},
}
np.save(os.path.join(SAVE_DIR, SAVE_NAME_V2), data, allow_pickle=True)
print("saved", data["train"]["X"].shape, data["test"]["X"].shape)

In [ ]:
d = np.load(os.path.join(SAVE_DIR, SAVE_NAME_V2), allow_pickle=True).item()
for s in ["train", "test"]:
    X, y = d[s]["X"], d[s]["y"]
    print(s, X.shape, "seizure fraction", round(float(y.mean()), 3))

In [ ]:
for s in ["train", "test"]:
    p, f, t = d[s]["patient"], d[s]["file"], d[s]["t0"]
    print(s, "lengths match:", len(p) == len(f) == len(t) == len(d[s]["X"]))
    print(s, "sample:", p[0], f[0], t[0])
    print(s, "unique patients:", sorted(set(p)))